# GAN solver — training, discriminator features, and move selection

The GAN is the first model in this study with an **explicit line-clear
objective**. A generator proposes `(rotation, position)` from board + piece +
noise; a discriminator judges `(pre-board, post-board, action)` with a realism
logit **and** a 5-way line-clear head (0–4 lines).

This notebook steps through training, then visualizes **why a move is selected**:
generator proposals, discriminator features (CNN maps + action vector), and the
score terms that rank legal placements.

| | |
|---|---|
| **Idea** | Adversarial generator + discriminator with line-clear head |
| **Play** | Sample / enumerate legal placements, rank by realism − λ·lines |
| **Headline result** | Clears a few lines (~6) — works, high variance |

Shipped weights: `models/gan_clean_v2/`. Full training: `uv run train-gan`.


## 0. Setup


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

nb_dir = Path.cwd()
if not (nb_dir / "nb_setup.py").exists():
    nb_dir = nb_dir / "notebooks"
sys.path.insert(0, str(nb_dir))

import nb_setup as N
N.setup_notebook()

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from puzzlegrid.pieces import PieceType
from puzzlegrid import placement
from gan_friendly_game import (
    encode_board_tensor,
    simulate_placement_detailed,
    snap_to_valid,
    piece_type_to_idx,
    GANFriendlyGame,
)
from replay_gan import load_gan_bundle, build_action_batch, PuzzleGridGANPlayer
from training_gan import (
    build_tetris_gan,
    extract_board_and_actions,
    load_training_data,
    subsample_training_data,
)

print("TF", tf.__version__)


## 1. Why a GAN after the VAE

The VAE reconstructs demonstrated placements and has **no value signal**. The
discriminator adds two things the VAE lacked:

1. **Realism** — does this `(pre, post, action)` look like a teacher move?
2. **Line-clear classification** — 0 / 1 / 2 / 3 / 4 lines, trained on real
   clears and on fakes labeled 0.

The generator is warmed up with supervised rotation/position loss, then trained
adversarially with a line-clear *reward* term. See `docs/readme_gan.md` and
`docs/VAE_TO_GAN_TRANSITION_RESULTS.md`.


## 2. Features the GAN actually trains on


In [ ]:
games = load_training_data(str(N.DATA_DIR), max_files=3)
data = extract_board_and_actions(games)
print("samples", len(data["board"]))
print("action vector layout: piece(7) + rotation(4) + pos(2) = 13")
print("line-clear histogram:", dict(zip(*np.unique(data["line_clears"].astype(int), return_counts=True))))

# Show a real pre-board and the post-board after the recorded placement.
from training_gan import precompute_post_boards
tiny = {k: v[:8] for k, v in data.items()}
post = precompute_post_boards(tiny)
i = int(np.argmax(tiny["line_clears"]))
pre = tiny["board"][i]
N.show_boards(
    [
        {"board": (pre > 0).astype(int).tolist(),
         "title": f"pre-board  piece={N.PIECE_NAMES[int(tiny['piece_idx'][i])]}"},
        {"board": (post[i, ..., 0] > 0).astype(int).tolist(),
         "title": f"post-board  lines={int(tiny['line_clears'][i])}"},
    ],
    cols=2,
)


## 3. Architecture

```
Generator
  board → CNN 256   piece → Dense 64   noise (32)
       concat → MLP 512 + 2× residual
       → rotation logits (4)  and  position sigmoid (2)

Discriminator
  pre-board → CNN 256
  post-board → same CNN 256
  action (13)
       concat → MLP 512 + 2× residual
       → realism logit (1)   line_clear logits (5)
```

Both CNNs reuse the VAE v2 board encoder recipe (`build_board_encoder`).


In [ ]:
gan_fresh = build_tetris_gan()
print("generator inputs", [t.name for t in gan_fresh.generator.inputs])
print("generator outputs", [t.name for t in gan_fresh.generator.outputs])
print("disc inputs", [t.name for t in gan_fresh.discriminator.inputs])
print("disc outputs", [t.name for t in gan_fresh.discriminator.outputs])
gan_fresh.generator.summary()


## 4. Optional short training

Default uses shipped `models/gan_clean_v2/`. A 1-epoch warmup on a few hundred
rows is enough to *show* the API; it will not reproduce the published ~6-line
player. Full run: `uv run train-gan`.


In [ ]:
RUN_SHORT_TRAINING = False

if RUN_SHORT_TRAINING:
    from training_gan import supervised_generator_warmup, train_gan
    games = load_training_data(str(N.DATA_DIR), max_files=3)
    data = subsample_training_data(extract_board_and_actions(games), 256)
    gan = build_tetris_gan(line_clear_reward_weight=6.0)
    gan.compile()
    train_gan(
        gan, data,
        epochs=1, warmup_epochs=1, batch_size=32,
        model_dir=str(N.REPO_ROOT / "models" / "gan_notebook_demo"),
        max_samples_for_post=256,
    )
else:
    print("Skipping in-notebook training.")


## 5. Load the shipped GAN


In [ ]:
generator, discriminator, gan_cfg = load_gan_bundle(str(N.GAN_DIR))
print(gan_cfg)
noise_dim = int(gan_cfg.get("noise_dim", 32))
print("noise_dim", noise_dim, "line_clear_reward_weight", gan_cfg.get("line_clear_reward_weight"))

# Nested CNN inside generator / discriminator (VAE-v2 board encoder recipe).
gen_cnn = generator.get_layer("board_encoder_v2")
disc_cnn = discriminator.get_layer("board_encoder_v2")
conv_names = ["conv2d", "conv2d_1", "conv2d_2", "conv2d_3"]
gen_acts = N.conv_activation_model(gen_cnn, conv_names)
disc_acts = N.conv_activation_model(disc_cnn, conv_names)
print("gen cnn", gen_cnn.name, "disc cnn", disc_cnn.name)


## 6. Experiment lab — piece and scenario

Edit and re-run. Useful contrasts:

- `almost_line` + `I` or `O` — does the line-clear head fire?
- `right_well` + `I` — Tetris setup; generator vs discriminator disagreement
- `bumpy` + `T` — messy surface, where realism and clears fight


In [ ]:
PIECE = "I"
SCENARIO = "almost_line"
LINE_CLEAR_BONUS = 1.5   # replay_gan default; try 0.0 vs 6.0

board = N.SCENARIOS[SCENARIO]()
piece_type = PieceType[PIECE]
candidates = placement.get_valid_placements(board, piece_type)
print(f"{SCENARIO}/{PIECE}: {len(candidates)} legal placements")
N.show_board(board, title=f"{SCENARIO} — piece {PIECE}")


## 7. Generator proposals under noise

The generator is stochastic in noise $z$. Sample many noises, snap each
decoded `(row, col, rot)` onto the legal set, and see the **proposal cloud**.
If proposals pile up on one column, the generator has collapsed; if they miss
the well, the discriminator has to save the play.


In [ ]:
board_in = encode_board_tensor(board)
piece_oh = N.piece_onehot(PIECE)
n_noise = 64
noise = np.random.randn(n_noise, noise_dim).astype(np.float32)
board_b = np.repeat(board_in, n_noise, axis=0)
piece_b = np.repeat(piece_oh, n_noise, axis=0)
rot_logits, pos = generator.predict([board_b, piece_b, noise], verbose=0)
rots = np.argmax(rot_logits, axis=-1)
rows = np.clip(np.rint(pos[:, 0] * 20).astype(int), 0, 19)
cols = np.clip(np.rint(pos[:, 1] * 10).astype(int), 0, 9)

snapped = []
for r, c, rot in zip(rows, cols, rots):
    s, dist = snap_to_valid(int(r), int(c), int(rot), candidates)
    snapped.append((s, dist, int(r), int(c), int(rot)))

from collections import Counter
legal_counts = Counter(s for s, *_ in snapped if s is not None)
print("unique snapped placements", len(legal_counts))
print("top proposals:", legal_counts.most_common(8))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].hist(cols, bins=np.arange(11) - 0.5, color="#67e8f9", edgecolor="#0b1220")
axes[0].set_title("raw generator columns")
axes[0].set_xlabel("col")
axes[1].hist(rots, bins=np.arange(5) - 0.5, color="#c084fc", edgecolor="#0b1220")
axes[1].set_title("raw generator rotations")
axes[1].set_xlabel("rot")
plt.tight_layout(); plt.show()

# Overlay proposal density on the board (count of snaps per cell of the piece).
density = np.zeros((20, 10), dtype=float)
for (s, dist, *_rest) in snapped:
    if s is None:
        continue
    for rr, cc in N.piece_cells(PIECE, *s):
        if 0 <= rr < 20 and 0 <= cc < 10:
            density[rr, cc] += 1.0
fig, ax = plt.subplots(figsize=(4.2, 7.4))
N.show_board(board, heatmap=density, title="generator proposal density (snapped)", ax=ax, show=False)
plt.tight_layout(); plt.show()


## 8. Discriminator scoring — the features that pick the move

For **every legal placement** we simulate the post-board and ask the
discriminator for:

- `realism` — raw logit (higher = more teacher-like)
- `p(lines)` — softmax over 0–4
- simulated `lines` from the engine (ground truth for this candidate)

Replay combines them as in `replay_gan.py`:

```
combined = realism - λ * simulated_lines - 0.3 * argmax(p(lines)) + 0.1 * snap_dist
```

then prefers more simulated lines, breaking ties with lower `combined`.


In [ ]:
def score_candidates(board, piece_name, line_clear_bonus=1.5):
    piece = PieceType[piece_name]
    cands = placement.get_valid_placements(board, piece)
    h, w = len(board), len(board[0])
    piece_idx = piece_type_to_idx(piece)
    pre = encode_board_tensor(board)
    posts, actions, lines, valids, post_boards = [], [], [], [], []
    for (r, c, rot) in cands:
        sim = simulate_placement_detailed(board, piece, r, c, rot, snap=False)
        action = build_action_batch(piece_idx, rot, r, c, h, w)
        post_b = sim.post_board if sim.valid else board
        posts.append(encode_board_tensor(post_b)[0])
        actions.append(action[0])
        lines.append(int(sim.lines_cleared) if sim.valid else 0)
        valids.append(sim.valid)
        post_boards.append(post_b)
    pre_b = np.repeat(pre, len(cands), axis=0)
    realism, lc_logits = discriminator.predict(
        [pre_b, np.asarray(posts), np.asarray(actions)], verbose=0
    )
    rows = []
    for i, (r, c, rot) in enumerate(cands):
        lc_prob = tf.nn.softmax(lc_logits[i]).numpy()
        lc_pred = int(np.argmax(lc_prob))
        realism_v = float(realism[i, 0])
        combined = realism_v - line_clear_bonus * lines[i] - 0.3 * lc_pred
        rows.append({
            "row": r, "col": c, "rot": rot,
            "realism": realism_v,
            "lc_pred": lc_pred,
            "p_clear": float(1.0 - lc_prob[0]),
            "lines": lines[i],
            "combined": combined,
            "valid": valids[i],
            "post": post_boards[i],
            "lc_prob": lc_prob,
            "action": np.asarray(actions[i]),
        })
    rows.sort(key=lambda d: (-d["lines"], d["combined"]))
    return rows

ranked = score_candidates(board, PIECE, line_clear_bonus=LINE_CLEAR_BONUS)
print(f"{len(ranked)} candidates")
N.ranked_table(
    ranked,
    ["combined", "lines", "realism", "lc_pred", "p_clear", "col", "row", "rot"],
    n=15,
)


In [ ]:
pick, miss = ranked[0], ranked[-1]
N.show_boards(
    [
        {"board": board, "piece": (pick["row"], pick["col"], pick["rot"]),
         "piece_type": PIECE,
         "title": f"GAN pick  lines={pick['lines']}  real={pick['realism']:.2f}"},
        {"board": pick["post"],
         "title": f"post-board after pick  p(clear)={pick['p_clear']:.2f}"},
        {"board": board, "piece": (miss["row"], miss["col"], miss["rot"]),
         "piece_type": PIECE,
         "title": f"lowest-ranked  lines={miss['lines']}  real={miss['realism']:.2f}"},
    ],
    cols=3,
)


## 9. Which feature decided?

Decompose the ranking into the terms the player actually uses. If flipping
`LINE_CLEAR_BONUS` changes the pick, the line-clear head (plus simulated lines)
is doing the work; if not, realism is dominating — the VAE failure mode again.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
xs = np.arange(min(20, len(ranked)))
sub = ranked[: len(xs)]
axes[0].bar(xs, [r["realism"] for r in sub], color="#60a5fa")
axes[0].set_title("realism logit")
axes[1].bar(xs, [r["lines"] for r in sub], color="#4ade80", label="sim lines")
axes[1].plot(xs, [r["lc_pred"] for r in sub], "o-", color="#facc15", label="pred lines")
axes[1].set_title("line-clear features"); axes[1].legend(fontsize=8)
axes[2].bar(xs, [r["combined"] for r in sub], color="#f87171")
axes[2].set_title("combined (lower/ties after lines)")
for ax in axes:
    ax.set_xlabel("candidate rank")
plt.tight_layout(); plt.show()

# Column heatmap of predicted P(clear) — does the head see the well / gap?
heat = np.full((20, 10), np.nan)
for r in ranked:
    rr, cc = r["row"], r["col"]
    val = r["p_clear"]
    if np.isnan(heat[rr, cc]) or val > heat[rr, cc]:
        heat[rr, cc] = val
fig, ax = plt.subplots(figsize=(4.2, 7.4))
N.show_board(board, heatmap=heat, title="max P(clear) at each resting cell", ax=ax, show=False)
plt.tight_layout(); plt.show()

# Ablate the line-clear bonus and see if the pick changes.
ranked_realism = score_candidates(board, PIECE, line_clear_bonus=0.0)
print("pick with λ=LINE_CLEAR_BONUS", (pick["row"], pick["col"], pick["rot"]), "lines", pick["lines"])
print("pick with λ=0 (realism only)",
      (ranked_realism[0]["row"], ranked_realism[0]["col"], ranked_realism[0]["rot"]),
      "lines", ranked_realism[0]["lines"])


## 10. CNN features of pre- vs post-board

The discriminator shares one CNN across pre- and post-boards. Comparing feature
maps for the **chosen** post-board vs a **no-clear** post-board shows whether
later conv channels respond to completed rows.


In [ ]:
pre_maps = disc_acts.predict(encode_board_tensor(board), verbose=0)
post_maps = disc_acts.predict(encode_board_tensor(pick["post"]), verbose=0)
N.show_feature_maps(pre_maps[0], "pre-board conv2d (32 maps, first 16)", max_channels=16)
N.show_feature_maps(post_maps[0], "post-board conv2d after GAN pick", max_channels=16)
# Deepest spatial map (5×5×128) — more semantic, less literal occupancy.
N.show_feature_maps(pre_maps[3], "pre-board conv2d_3 (5×5)", max_channels=16)
N.show_feature_maps(post_maps[3], "post-board conv2d_3 (5×5)", max_channels=16)


### Action-vector features

The 13-d action is `piece(7) | rotation(4) | (row/H, col/W)`. Plotting it for
the pick vs a rejected move makes the discriminator’s non-board inputs visible.


In [ ]:
labels = [f"P{n}" for n in N.PIECE_NAMES] + ["R0", "R1", "R2", "R3", "row", "col"]
fig, ax = plt.subplots(figsize=(10, 3.2))
x = np.arange(13)
ax.bar(x - 0.2, pick["action"], 0.4, label="GAN pick", color="#67e8f9")
ax.bar(x + 0.2, miss["action"], 0.4, label="low-ranked", color="#f87171")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_title("action vector (13-d) — what the discriminator sees besides the boards")
ax.legend()
plt.tight_layout(); plt.show()


## 11. Solver: a few in-process moves

`PuzzleGridGANPlayer` can run without the API (`use_inprocess=True`). Play a
short game on this scenario’s piece set and watch lines.


In [ ]:
player = PuzzleGridGANPlayer(
    str(N.GAN_DIR),
    use_inprocess=True,
    num_samples=8,
    line_clear_bonus=LINE_CLEAR_BONUS,
)
# Start from an empty game with a biased sequence so the scenario can appear.
seq = [PIECE, "O", "T", "I", "L", "J", "S", "Z"] * 8
player.create_game(piece_sequence=seq)
# Force the first board into our scenario by copying into the local game.
player._local_game._game.board.grid = [row[:] for row in board]
state = player.play_game(max_moves=12, mode="search", delay=0.0)
print("status", state.get("status"),
      "lines", state.get("lines_cleared"),
      "pieces", state.get("pieces_placed"),
      "score", state.get("score"))
N.show_board(state["board"], title=f"after 12 GAN moves  lines={state.get('lines_cleared')}")


## 12. Takeaways

- The GAN’s **selection rule is a linear mix of learned features** (realism,
  predicted clears) and an engine feature (simulated lines). The plots above
  show which term wins on the board you chose.
- Generator noise explores; the discriminator (plus snap-to-legal) **chooses**.
  Ranking legal candidates with a line-clear head is already halfway to the
  transformer — but the GAN still *generates* then patches, instead of scoring
  the full legal set as its primary policy.
- Feature maps of pre vs post are the right place to look for “did it notice
  the completed row?”
- Ceiling in this study: ~6 lines. Next notebook: score every candidate with a
  transformer.

Next: `03_transformer_training_and_solver.ipynb`.
